# Assignment: Improving Neural Networks
## Regularization, Optimization & Hyperparameter Tuning | PyTorch

---

### Assignment Objectives

By completing this assignment you will:

1. **Diagnose** bias vs variance problems by analyzing training and validation loss curves.
2. **Implement** regularization techniques (L2 weight decay, Dropout) and observe their effect on overfitting.
3. **Compare** optimizers (SGD with momentum, Adam, AdamW) in terms of convergence speed and final performance.
4. **Implement** learning rate schedules (StepLR, CosineAnnealing, ReduceLROnPlateau) and analyze their impact.
5. **Conduct** systematic hyperparameter tuning using random search over architectures, learning rates, dropout, and weight decay.
6. **Deploy** a best model combining all improvements, demonstrating measurable gains over the baseline.

> **Deliverable:** After running every cell, you will have all the figures, tables, and analysis needed to write a professional lab report.

---

### Report Structure Guide

Your report should follow this structure (each section maps to a part of this notebook):

| Report Section | What to Write | Notebook Section |
|---|---|---|
| **1. Introduction** | Bias/variance, regularization, optimization — why do networks need improvement? | Part 1 |
| **2. Dataset** | Describe the Ames Housing dataset, features, preprocessing | Part 3 |
| **3. Baseline** | Baseline model architecture, training curves, evidence of overfitting | Part 4 |
| **4. Regularization** | L2 and Dropout experiments, effect on train/val gap | Parts 5-6 |
| **5. Optimization** | Optimizer comparison, LR scheduling, BatchNorm | Parts 7-9 |
| **6. Tuning** | Random search setup, results table, best configuration | Part 10 |
| **7. Best Model** | Final model with all improvements, performance comparison | Part 11 |
| **8. Conclusion** | Summary, improvement over baseline, lessons learned | Part 12 |

### Grading Rubric

| Category | Points | Description |
|---|---|---|
| Baseline Model | 15 | Correct implementation, training curves, overfitting diagnosis |
| Regularization | 20 | L2 and Dropout experiments with analysis |
| Optimizers | 20 | SGD/Adam/AdamW comparison with convergence analysis |
| LR Scheduling | 15 | Three schedules implemented and compared |
| Hyperparameter Tuning | 20 | Systematic random search with results table |
| Report | 10 | Clear writing, figures, analysis, conclusions |
| **Total** | **100** | |

---

*Apeiron AI | "Boundless Possibilities, Infinite Potential"*
*© 2026 | www.aperionaiml.com*

---

# Part 1: Background — Improving Deep Networks

## 1.1 The Bias-Variance Tradeoff

After training a neural network, there are two fundamental failure modes:

```
     High Bias (Underfitting)          Good Fit              High Variance (Overfitting)
    ┌────────────────┐    ┌────────────────┐    ┌────────────────┐
    │ Train loss: HIGH  │    │ Train loss: LOW  │    │ Train loss: LOW  │
    │ Val loss:   HIGH  │    │ Val loss:   LOW  │    │ Val loss:   HIGH │
    │ Gap:        SMALL │    │ Gap:        SMALL│    │ Gap:        LARGE│
    └────────────────┘    └────────────────┘    └────────────────┘
     Model too simple              Just right              Model memorizes noise
```

## 1.2 The Improvement Recipe

```
   Start with baseline
         │
         ▼
   ┌─────────────────────┐
   │ Diagnose the problem │ ──► Train & val loss curves
   └──────────┬──────────┘
              │
      ┌──────┴──────┐
      ▼              ▼
   Underfitting?    Overfitting?
      │              │
      ▼              ▼
   • Bigger model   • L2 regularization
   • Train longer   • Dropout
   • Lower reg.     • Data augmentation
                     • Early stopping
         │
         ▼
   ┌─────────────────────┐
   │ Optimize training    │ ──► Better optimizer, LR schedule, BatchNorm
   └──────────┬──────────┘
              │
              ▼
   ┌─────────────────────┐
   │ Tune hyperparameters │ ──► Random search over architecture, LR, reg.
   └─────────────────────┘
```

## 1.3 Key Techniques

| Technique | What It Does | When to Use |
|---|---|---|
| **L2 Regularization** | Adds λ‖w‖² to loss, penalizes large weights | Overfitting (large train/val gap) |
| **Dropout** | Randomly zeros p% of neurons during training | Overfitting, especially with large models |
| **Momentum** | Smooths gradient updates: v = βv + (1-β)g | Noisy gradients, slow convergence |
| **Adam** | Adaptive per-parameter learning rates | Default choice for most problems |
| **AdamW** | Adam + decoupled weight decay | When combining Adam with L2 |
| **BatchNorm** | Normalizes activations to zero mean, unit variance | Unstable training, helps convergence |
| **LR Scheduling** | Reduces learning rate during training | Fine-tuning, avoiding overshooting |
| **Early Stopping** | Stop when validation loss stops improving | Prevent overfitting |

---

# Part 2: Environment Setup

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  INSTALL DEPENDENCIES (run once)                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
%pip install torch numpy matplotlib pandas scikit-learn seaborn --quiet


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  IMPORTS                                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import time
import copy

# ── Plotting defaults ────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})
sns.set_style("whitegrid")

# ── Device selection ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
if device.type == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

# ── Reproducibility ──────────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Random seed     : {SEED}")


---

# Part 3: Ames Housing Dataset

## 3.1 About the Dataset

The **Ames Housing** dataset describes residential properties in Ames, Iowa. It contains ~1,460 houses with 80 features (we use numeric features only). The target is **SalePrice** — the sale price of the house in dollars.

This is a **different dataset** from Assignment 1 (which used California Housing). Ames Housing has more features, more variability, and is more challenging to predict.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LOAD AMES HOUSING DATASET                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

from sklearn.datasets import fetch_openml

try:
    ames = fetch_openml(name="house_prices", version=1, as_frame=True, parser="auto")
    df = ames.data.copy()
    df["SalePrice"] = ames.target.astype(float)
    dataset_name = "Ames Housing (house_prices)"
except Exception as e:
    print(f"Ames Housing not available ({e}). Using California Housing as fallback.")
    from sklearn.datasets import fetch_california_housing
    cal = fetch_california_housing(as_frame=True)
    df = cal.data.copy()
    df["SalePrice"] = cal.target * 100000  # scale to dollar-like range
    dataset_name = "California Housing (fallback)"

print(f"Dataset: {dataset_name}")
print(f"Shape:   {df.shape}")

# Select numeric columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
target_col = "SalePrice"
feature_cols = [c for c in numeric_cols if c != target_col]

print(f"Numeric features: {len(feature_cols)}")
print(f"Target: {target_col}")
df[feature_cols + [target_col]].head()


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  PREPROCESS AND SPLIT                                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Handle missing values: fill with median
X_df = df[feature_cols].fillna(df[feature_cols].median())
y_vals = df[target_col].values.astype(np.float32)

# Normalize features
X_np = X_df.values.astype(np.float32)
X_mean = X_np.mean(axis=0)
X_std  = X_np.std(axis=0) + 1e-8  # prevent division by zero
X_norm = (X_np - X_mean) / X_std

# Normalize target (log-transform for house prices, then standardize)
y_log = np.log1p(y_vals)
y_mean = y_log.mean()
y_std_val = y_log.std()
y_norm = (y_log - y_mean) / y_std_val

# Split: 70% train, 15% val, 15% test
n = len(X_norm)
idx = np.random.permutation(n)
n_train = int(0.70 * n)
n_val   = int(0.85 * n)

X_train_np = X_norm[idx[:n_train]]
X_val_np   = X_norm[idx[n_train:n_val]]
X_test_np  = X_norm[idx[n_val:]]
y_train_np = y_norm[idx[:n_train]]
y_val_np   = y_norm[idx[n_train:n_val]]
y_test_np  = y_norm[idx[n_val:]]

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train_np, dtype=torch.float32)
X_val_t   = torch.tensor(X_val_np,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test_np,  dtype=torch.float32)
y_train_t = torch.tensor(y_train_np, dtype=torch.float32).unsqueeze(1)
y_val_t   = torch.tensor(y_val_np,   dtype=torch.float32).unsqueeze(1)
y_test_t  = torch.tensor(y_test_np,  dtype=torch.float32).unsqueeze(1)

# DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t),     batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t),   batch_size=BATCH_SIZE, shuffle=False)

n_features = X_train_t.shape[1]
print(f"Number of features: {n_features}")
print(f"Train: {X_train_t.shape[0]}, Val: {X_val_t.shape[0]}, Test: {X_test_t.shape[0]}")


> **For your report:** Describe the dataset. How many samples, how many features? Why did we log-transform the target? Why normalize?

---

# Part 4: Baseline Model (No Regularization)

We start with a large model and NO regularization to deliberately cause overfitting.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  BASELINE MODEL (no regularization)                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class BaselineMLP(nn.Module):
    def __init__(self, n_input, hidden_sizes=[256, 128, 64]):
        super().__init__()
        layers = []
        prev = n_input
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def train_model(model, train_loader, val_loader, optimizer, epochs=100,
                scheduler=None, verbose=True):
    """Generic training loop. Returns history dict."""
    model.to(device)
    criterion = nn.MSELoss()
    history = {"train_loss": [], "val_loss": [], "lr": []}

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        train_loss_sum = 0.0
        n_batches = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item()
            n_batches += 1

        # --- Validate ---
        model.eval()
        val_loss_sum = 0.0
        n_val = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_pred = model(X_batch)
                val_loss_sum += criterion(y_pred, y_batch).item()
                n_val += 1

        train_loss = train_loss_sum / n_batches
        val_loss = val_loss_sum / max(n_val, 1)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(current_lr)

        if scheduler is not None:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        if verbose and (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:4d}/{epochs}  |  "
                  f"Train: {train_loss:.6f}  |  Val: {val_loss:.6f}  |  LR: {current_lr:.6f}")

    return history

# --- Train baseline ---
baseline = BaselineMLP(n_features, [256, 128, 64])
print(baseline)
print(f"\nParameters: {sum(p.numel() for p in baseline.parameters()):,}")

baseline_opt = optim.Adam(baseline.parameters(), lr=0.001)
baseline_hist = train_model(baseline, train_loader, val_loader, baseline_opt, epochs=100)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  BASELINE TRAINING CURVES                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(baseline_hist["train_loss"], label="Train Loss", linewidth=2)
ax.plot(baseline_hist["val_loss"], label="Val Loss", linewidth=2, linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Baseline Model: Training vs Validation Loss", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate the gap
final_train = baseline_hist["train_loss"][-1]
final_val = baseline_hist["val_loss"][-1]
ax.annotate(f"Gap = {final_val - final_train:.4f}",
            xy=(len(baseline_hist["train_loss"])-1, final_val),
            fontsize=11, fontweight="bold", color="red")

plt.tight_layout()
plt.savefig("asgn_fig_baseline_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nFinal Train Loss: {final_train:.6f}")
print(f"Final Val Loss:   {final_val:.6f}")
print(f"Gap (val - train): {final_val - final_train:.6f}")
print("\nA large gap indicates OVERFITTING!")


> **For your report:** Look at the training curves. Is there evidence of overfitting (high variance)? How large is the gap between training and validation loss?

---

# Part 5: L2 Regularization (Weight Decay)

L2 regularization adds a penalty term to the loss: **L_total = L_data + λ ‖w‖²**

This discourages large weights and reduces overfitting. In PyTorch, this is the `weight_decay` parameter.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  L2 REGULARIZATION EXPERIMENT                                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

weight_decays = [0, 1e-4, 1e-3, 1e-2]
l2_results = {}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for wd in weight_decays:
    torch.manual_seed(SEED)
    model = BaselineMLP(n_features, [256, 128, 64])
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=wd)
    hist = train_model(model, train_loader, val_loader, opt, epochs=100, verbose=False)
    l2_results[f"wd={wd}"] = hist

    axes[0].plot(hist["train_loss"], label=f"wd={wd} (train)")
    axes[1].plot(hist["val_loss"], label=f"wd={wd} (val)")

axes[0].set_title("Training Loss", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title("Validation Loss", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MSE"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("L2 Regularization (Weight Decay) Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("asgn_fig_l2_regularization.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nL2 Regularization Results:")
print(f"  {'Weight Decay':<15s}  {'Final Train':>12s}  {'Final Val':>12s}  {'Gap':>10s}")
print("  " + "-" * 55)
for name, hist in l2_results.items():
    tr = hist["train_loss"][-1]
    va = hist["val_loss"][-1]
    print(f"  {name:<15s}  {tr:12.6f}  {va:12.6f}  {va-tr:10.6f}")


> **For your report:** How does weight decay affect the train/val gap? What happens with too much regularization (wd=0.01)? Is there a sweet spot?

---

# Part 6: Dropout

Dropout randomly sets a fraction of neuron outputs to zero during training. This prevents co-adaptation of neurons and acts as an ensemble of sub-networks.

**Critical:** Use `model.train()` for training (dropout ON) and `model.eval()` for evaluation (dropout OFF).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  DROPOUT EXPERIMENT                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class DropoutMLP(nn.Module):
    def __init__(self, n_input, hidden_sizes=[256, 128, 64], dropout_rate=0.3):
        super().__init__()
        layers = []
        prev = n_input
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

dropout_rates = [0.0, 0.3, 0.5]
dropout_results = {}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for dr in dropout_rates:
    torch.manual_seed(SEED)
    model = DropoutMLP(n_features, [256, 128, 64], dropout_rate=dr)
    opt = optim.Adam(model.parameters(), lr=0.001)
    hist = train_model(model, train_loader, val_loader, opt, epochs=100, verbose=False)
    dropout_results[f"dropout={dr}"] = hist

    axes[0].plot(hist["train_loss"], label=f"p={dr} (train)")
    axes[1].plot(hist["val_loss"], label=f"p={dr} (val)")

axes[0].set_title("Training Loss", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title("Validation Loss", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MSE"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("Dropout Rate Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("asgn_fig_dropout.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nDropout Results:")
print(f"  {'Dropout Rate':<15s}  {'Final Train':>12s}  {'Final Val':>12s}  {'Gap':>10s}")
print("  " + "-" * 55)
for name, hist in dropout_results.items():
    tr = hist["train_loss"][-1]
    va = hist["val_loss"][-1]
    print(f"  {name:<15s}  {tr:12.6f}  {va:12.6f}  {va-tr:10.6f}")


> **For your report:** How does dropout reduce overfitting? Why must you call `model.train()` vs `model.eval()`? What dropout rate works best?

---

# Part 7: Optimizer Comparison

We compare three optimizers on the same model and data.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  OPTIMIZER COMPARISON                                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

optimizers_config = {
    "SGD (lr=0.01, mom=0.9)": lambda p: optim.SGD(p, lr=0.01, momentum=0.9),
    "Adam (lr=0.001)":        lambda p: optim.Adam(p, lr=0.001),
    "AdamW (lr=0.001, wd=0.01)": lambda p: optim.AdamW(p, lr=0.001, weight_decay=0.01),
}

opt_results = {}
opt_times = {}

fig, ax = plt.subplots(figsize=(10, 5))

for name, opt_fn in optimizers_config.items():
    torch.manual_seed(SEED)
    model = BaselineMLP(n_features, [256, 128, 64])
    opt = opt_fn(model.parameters())

    t0 = time.time()
    hist = train_model(model, train_loader, val_loader, opt, epochs=100, verbose=False)
    opt_times[name] = time.time() - t0
    opt_results[name] = hist

    ax.plot(hist["val_loss"], label=name, linewidth=2)

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation MSE")
ax.set_title("Optimizer Comparison — Validation Loss", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("asgn_fig_optimizer_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Training time comparison
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(list(opt_times.keys()), list(opt_times.values()), color=["steelblue", "coral", "seagreen"])
ax.set_xlabel("Training Time (seconds)")
ax.set_title("Training Time Comparison", fontweight="bold")
plt.tight_layout()
plt.savefig("asgn_fig_optimizer_time.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nOptimizer Results:")
print(f"  {'Optimizer':<30s}  {'Final Val Loss':>14s}  {'Time (s)':>10s}")
print("  " + "-" * 60)
for name in optimizers_config:
    val = opt_results[name]["val_loss"][-1]
    t = opt_times[name]
    print(f"  {name:<30s}  {val:14.6f}  {t:10.2f}")


> **For your report:** Which optimizer converges fastest? Which achieves the lowest validation loss? Why does Adam/AdamW typically outperform vanilla SGD?

---

# Part 8: Learning Rate Scheduling

A fixed learning rate is rarely optimal. Schedules reduce the LR during training for finer convergence.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LR SCHEDULING EXPERIMENT                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

EPOCHS_LR = 100

schedules = {
    "No Schedule": None,
    "StepLR (step=20, γ=0.5)": lambda opt: optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.5),
    "CosineAnnealing": lambda opt: optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_LR),
    "ReduceOnPlateau": lambda opt: optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5),
}

lr_results = {}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, sched_fn in schedules.items():
    torch.manual_seed(SEED)
    model = BaselineMLP(n_features, [256, 128, 64])
    opt = optim.Adam(model.parameters(), lr=0.001)
    scheduler = sched_fn(opt) if sched_fn is not None else None

    hist = train_model(model, train_loader, val_loader, opt,
                       epochs=EPOCHS_LR, scheduler=scheduler, verbose=False)
    lr_results[name] = hist

    axes[0].plot(hist["val_loss"], label=name, linewidth=2)
    axes[1].plot(hist["lr"], label=name, linewidth=2)

axes[0].set_title("Validation Loss", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE"); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_title("Learning Rate Over Epochs", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Learning Rate"); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.suptitle("Learning Rate Schedule Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("asgn_fig_lr_scheduling.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nLR Schedule Results:")
print(f"  {'Schedule':<30s}  {'Final Val Loss':>14s}")
print("  " + "-" * 48)
for name, hist in lr_results.items():
    print(f"  {name:<30s}  {hist['val_loss'][-1]:14.6f}")


> **For your report:** Which LR schedule produced the best results? When does LR scheduling help most? What is the intuition behind cosine annealing?

---

# Part 9: Batch Normalization

BatchNorm normalizes activations within each mini-batch, stabilizing training and often allowing higher learning rates.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  BATCH NORMALIZATION EXPERIMENT                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class BatchNormMLP(nn.Module):
    def __init__(self, n_input, hidden_sizes=[256, 128, 64], dropout_rate=0.0):
        super().__init__()
        layers = []
        prev = n_input
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# Compare: with vs without BatchNorm
bn_configs = {
    "No BatchNorm": lambda: BaselineMLP(n_features, [256, 128, 64]),
    "With BatchNorm": lambda: BatchNormMLP(n_features, [256, 128, 64]),
}

fig, ax = plt.subplots(figsize=(10, 5))
bn_results = {}

for name, model_fn in bn_configs.items():
    torch.manual_seed(SEED)
    model = model_fn()
    opt = optim.Adam(model.parameters(), lr=0.001)
    hist = train_model(model, train_loader, val_loader, opt, epochs=100, verbose=False)
    bn_results[name] = hist
    ax.plot(hist["val_loss"], label=name, linewidth=2)

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation MSE")
ax.set_title("BatchNorm vs No BatchNorm", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("asgn_fig_batchnorm.png", dpi=150, bbox_inches="tight")
plt.show()

for name, hist in bn_results.items():
    print(f"  {name:<20s}  Final Val Loss: {hist['val_loss'][-1]:.6f}")


> **For your report:** Does BatchNorm help? Does the model with BatchNorm converge faster or to a lower loss? Explain why BatchNorm stabilizes training.

---

# Part 10: Systematic Hyperparameter Tuning

We use **random search** to explore the hyperparameter space. Random search is more efficient than grid search because it samples diverse combinations.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  RANDOM SEARCH HYPERPARAMETER TUNING                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import random
random.seed(SEED)

# Search space
search_space = {
    "hidden_size":  [64, 128, 256],
    "num_layers":   [2, 3, 4],
    "lr":           [1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    "dropout":      [0.0, 0.2, 0.5],
    "weight_decay": [0, 1e-4, 1e-3],
}

N_TRIALS = 12
tuning_results = []

print(f"Running {N_TRIALS} random hyperparameter configurations...")
print("=" * 80)

for trial in range(N_TRIALS):
    config = {k: random.choice(v) for k, v in search_space.items()}

    # Build model
    hidden = [config["hidden_size"]] * config["num_layers"]
    torch.manual_seed(SEED)
    model = BatchNormMLP(n_features, hidden, dropout_rate=config["dropout"])
    opt = optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    hist = train_model(model, train_loader, val_loader, opt, epochs=60, verbose=False)
    best_val = min(hist["val_loss"])

    config["best_val_loss"] = best_val
    config["final_val_loss"] = hist["val_loss"][-1]
    tuning_results.append(config)

    print(f"  Trial {trial+1:2d}: hidden={config['hidden_size']}, layers={config['num_layers']}, "
          f"lr={config['lr']:.4f}, drop={config['dropout']}, wd={config['weight_decay']}  "
          f"-> Val Loss = {best_val:.6f}")

# Sort by best validation loss
tuning_results.sort(key=lambda x: x["best_val_loss"])

print("\n" + "=" * 80)
print("\nTop 5 Configurations (sorted by best val loss):")
print(f"  {'Rank':<5s} {'Hidden':<8s} {'Layers':<8s} {'LR':<10s} {'Dropout':<9s} {'WD':<10s} {'Best Val':>10s}")
print("  " + "-" * 65)
for i, cfg in enumerate(tuning_results[:5]):
    print(f"  {i+1:<5d} {cfg['hidden_size']:<8d} {cfg['num_layers']:<8d} {cfg['lr']:<10.4f} "
          f"{cfg['dropout']:<9.1f} {cfg['weight_decay']:<10.5f} {cfg['best_val_loss']:10.6f}")

best_config = tuning_results[0]
print(f"\nBest configuration: {best_config}")


> **For your report:** What patterns do you see in the top configurations? Which hyperparameters matter most? Why is random search preferred over grid search?

---

# Part 11: Best Model — All Improvements Combined

We combine the best hyperparameters with BatchNorm, Dropout, AdamW, and Cosine Annealing.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TRAIN BEST MODEL WITH ALL IMPROVEMENTS                                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

BEST_EPOCHS = 200

# Use best config from random search
best_hidden = [best_config["hidden_size"]] * best_config["num_layers"]
best_lr = best_config["lr"]
best_dropout = best_config["dropout"]
best_wd = best_config["weight_decay"]

print(f"Best config: hidden={best_hidden}, lr={best_lr}, dropout={best_dropout}, wd={best_wd}")

torch.manual_seed(SEED)
best_model = BatchNormMLP(n_features, best_hidden, dropout_rate=best_dropout)
best_opt = optim.AdamW(best_model.parameters(), lr=best_lr, weight_decay=best_wd)
best_sched = optim.lr_scheduler.CosineAnnealingLR(best_opt, T_max=BEST_EPOCHS)

print(f"Parameters: {sum(p.numel() for p in best_model.parameters()):,}")
print(f"\nTraining for {BEST_EPOCHS} epochs with CosineAnnealing...")

# Train with early stopping
best_model.to(device)
criterion = nn.MSELoss()
best_val_loss = float("inf")
best_state = None
patience = 30
patience_counter = 0
best_hist = {"train_loss": [], "val_loss": [], "lr": []}

for epoch in range(BEST_EPOCHS):
    # Train
    best_model.train()
    train_sum = 0; nb = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        best_opt.zero_grad()
        loss = criterion(best_model(xb), yb)
        loss.backward()
        best_opt.step()
        train_sum += loss.item(); nb += 1

    # Val
    best_model.eval()
    val_sum = 0; nv = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_sum += criterion(best_model(xb), yb).item(); nv += 1

    tl = train_sum / nb; vl = val_sum / max(nv, 1)
    best_hist["train_loss"].append(tl)
    best_hist["val_loss"].append(vl)
    best_hist["lr"].append(best_opt.param_groups[0]["lr"])
    best_sched.step()

    if vl < best_val_loss:
        best_val_loss = vl
        best_state = copy.deepcopy(best_model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"  Early stopping at epoch {epoch+1}")
        break

    if (epoch + 1) % 40 == 0:
        print(f"  Epoch {epoch+1:4d}/{BEST_EPOCHS}  |  Train: {tl:.6f}  |  Val: {vl:.6f}")

# Load best weights
best_model.load_state_dict(best_state)
print(f"\nBest validation loss: {best_val_loss:.6f}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  BEST MODEL EVALUATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

best_model.eval()
with torch.no_grad():
    y_pred_test = best_model(X_test_t.to(device)).cpu().numpy().flatten()

# Convert back to original scale
y_pred_orig = np.expm1(y_pred_test * y_std_val + y_mean)
y_test_orig = np.expm1(y_test_np * y_std_val + y_mean)

# Metrics
mse_val  = np.mean((y_pred_orig - y_test_orig) ** 2)
rmse_val = np.sqrt(mse_val)
mae_val  = np.mean(np.abs(y_pred_orig - y_test_orig))
ss_res = np.sum((y_test_orig - y_pred_orig) ** 2)
ss_tot = np.sum((y_test_orig - np.mean(y_test_orig)) ** 2)
r2_val = 1 - ss_res / ss_tot

print("=" * 50)
print("  Best Model — Test Set Evaluation")
print("=" * 50)
print(f"  MSE:  {mse_val:,.0f}")
print(f"  RMSE: ${rmse_val:,.0f}")
print(f"  MAE:  ${mae_val:,.0f}")
print(f"  R²:   {r2_val:.4f}")
print("=" * 50)

# Predicted vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(y_test_orig, y_pred_orig, alpha=0.4, s=15, color="steelblue")
lims = [min(y_test_orig.min(), y_pred_orig.min()),
        max(y_test_orig.max(), y_pred_orig.max())]
ax.plot(lims, lims, "r--", linewidth=2, label="Perfect prediction")
ax.set_xlabel("Actual Price ($)")
ax.set_ylabel("Predicted Price ($)")
ax.set_title(f"Predicted vs Actual (R² = {r2_val:.4f})", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(best_hist["train_loss"], label="Train", linewidth=2)
ax.plot(best_hist["val_loss"], label="Val", linewidth=2, linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Best Model Training Curves", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("asgn_fig_best_model.png", dpi=150, bbox_inches="tight")
plt.show()


> **For your report:** How much did the best model improve over the baseline? Compare R², RMSE, and the train/val gap. Which improvements contributed most?

---

# Part 12: Save & Conclusion

## 12.1 Save the Best Model

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SAVE BEST MODEL                                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

save_path = "best_improved_model.pth"
torch.save({
    "model_state_dict": best_state,
    "config": best_config,
    "metrics": {"mse": float(mse_val), "rmse": float(rmse_val),
                "mae": float(mae_val), "r2": float(r2_val)},
}, save_path)

print(f"Best model saved to: {save_path}")
print(f"R² = {r2_val:.4f}")


## 12.2 Summary of All Saved Figures

Every figure generated by this notebook is saved as a PNG for your report:

| Figure File | Content | Report Section |
|---|---|---|
| `asgn_fig_baseline_curves.png` | Baseline train vs val loss (shows overfitting) | Baseline |
| `asgn_fig_l2_regularization.png` | Weight decay comparison (4 values) | Regularization |
| `asgn_fig_dropout.png` | Dropout rate comparison (3 values) | Regularization |
| `asgn_fig_optimizer_comparison.png` | SGD vs Adam vs AdamW validation curves | Optimization |
| `asgn_fig_optimizer_time.png` | Training time bar chart | Optimization |
| `asgn_fig_lr_scheduling.png` | LR schedule comparison + LR curves | Optimization |
| `asgn_fig_batchnorm.png` | BatchNorm vs no BatchNorm | Optimization |
| `asgn_fig_best_model.png` | Best model predictions + training curves | Best Model |

## 12.3 Report Writing Guide

### What to Write in Each Section

**1. Introduction (1 page)**
- What is the bias-variance tradeoff? What is overfitting?
- Why do neural networks need regularization and optimization?
- Overview of techniques: L2, Dropout, Adam, LR scheduling, BatchNorm

**2. Dataset (0.5 page)**
- Describe the Ames Housing dataset (or fallback dataset)
- Preprocessing: missing values, normalization, log-transform
- Train/val/test split rationale

**3. Baseline (0.5 page)**
- Architecture: input → 256 → 128 → 64 → 1
- Include `asgn_fig_baseline_curves.png`
- Diagnose: overfitting (large gap) or underfitting?

**4. Regularization (1 page)**
- L2: Include `asgn_fig_l2_regularization.png`, discuss sweet spot
- Dropout: Include `asgn_fig_dropout.png`, explain mechanism
- How do they reduce the train/val gap?

**5. Optimization (1 page)**
- Optimizers: Include `asgn_fig_optimizer_comparison.png`, convergence analysis
- LR Scheduling: Include `asgn_fig_lr_scheduling.png`, which works best?
- BatchNorm: Include `asgn_fig_batchnorm.png`, why it helps

**6. Hyperparameter Tuning (0.5 page)**
- Random search setup and results table
- Which hyperparameters matter most?

**7. Best Model (0.5 page)**
- Final configuration and metrics
- Include `asgn_fig_best_model.png`
- Improvement over baseline (quantify!)

**8. Conclusion (0.5 page)**
- Summary of key findings
- Which technique helped most?
- Lessons learned, potential further improvements

---

<center>

### Assignment Complete

**Remember to:**
1. Run all cells from top to bottom before submission
2. Ensure all outputs and figures are visible
3. Write your report using the generated figures
4. Save your best model file

---

*Apeiron AI | "Boundless Possibilities, Infinite Potential"*
*© 2026 | www.aperionaiml.com*

</center>